In [1]:
from datetime import datetime, timedelta
import uo_pyfetch
import calendar
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import requests
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from pyproj import Transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd
from IPython.display import HTML


In [2]:
bbox = [-1.652756, 54.973377, -1.620483, 54.983721]

In [3]:
print("\nFetching sensors in bbox...")
sensors_df = uo_pyfetch.get_sensors() #bondary box


Fetching sensors in bbox...


In [ ]:
print("\nFetching sensors in bbox...")
sensors_df = uo_pyfetch.get_sensors(limit=50)
HTML(sensors_df.head(10000).to_html())


In [ ]:
print("\nFetching sensor data (last 2 hours)...")
sensor_data_df = uo_pyfetch.get_sensor_data(
    last_n_hours=2,
    variables=["NO2", "PM10"],
    bbox=bbox,
    limit=-1
)
HTML(sensor_data_df.to_html(index=False))

In [ ]:


if not sensor_data_df.empty:
    sensor_name = sensor_data_df.iloc[0]["Sensor_Name"]
    print(f"\nFetching full history for sensor: {sensor_name}")
    
    # Define start/end
    start = datetime.now() - timedelta(weeks=260)  # ~5 years
    end = datetime.now()
    
    # Split into 1-month intervals to avoid timeout
    chunk_size = 30  # days per chunk
    current_start = start
    df_list = []

    while current_start < end:
        current_end = min(current_start + timedelta(days=chunk_size), end)
        print(f"Fetching from {current_start.date()} to {current_end.date()}")
        
        try:
            chunk_df = uo_pyfetch.get_sensor_data_by_name(
                sensor_name,
                start=current_start,
                end=current_end,
                variables=["NO2"]
            )
            if not chunk_df.empty:
                df_list.append(chunk_df)
        except Exception as e:
            print(f"Error fetching chunk {current_start} -> {current_end}: {e}")
        
        current_start = current_end  # move to next chunk

    # Combine all chunks
    if df_list:
        sensor_specific_df = pd.concat(df_list, ignore_index=True)
    else:
        sensor_specific_df = pd.DataFrame()
    
    HTML(sensor_specific_df.head().to_html())
else:
    print("\nNo sensors found in bbox.")


In [ ]:
HTML(sensor_specific_df.head(100000).to_html())

In [11]:
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import HTML

# Example bbox
bbox = [-1.652756, 54.973377, -1.620483, 54.983721]

# Total hours to fetch
total_hours = 8736*10  # 1 year
batch_hours = 168*4   # 1 week
variables = ["Congestion"]

# End time is now
end_time = datetime.utcnow()
sensor_data_list = []

print("\nFetching sensor data in batches...")

for start_offset in range(0, total_hours, batch_hours):
    start_time = end_time - timedelta(hours=start_offset + batch_hours)
    batch_end_time = end_time - timedelta(hours=start_offset)
    
    print(f"Fetching data from {start_time} to {batch_end_time}...")
    
    try:
        batch_df = uo_pyfetch.get_sensor_data(
            last_n_hours=batch_hours,
            variables=variables,
            bbox=bbox,
            limit=-1,
        )
        sensor_data_list.append(batch_df)
    except Exception as e:
        print(f"Error fetching batch: {e}")
        continue

# Combine all batches into one DataFrame
sensor_data_df = pd.concat(sensor_data_list, ignore_index=True)

# Display in HTML
HTML(sensor_data_df.to_html(index=False))



Fetching sensor data in batches...
Fetching data from 2025-10-18 20:45:21.720547 to 2025-11-15 20:45:21.720547...
Fetching data from 2025-09-20 20:45:21.720547 to 2025-10-18 20:45:21.720547...
Fetching data from 2025-08-23 20:45:21.720547 to 2025-09-20 20:45:21.720547...
Fetching data from 2025-07-26 20:45:21.720547 to 2025-08-23 20:45:21.720547...
Fetching data from 2025-06-28 20:45:21.720547 to 2025-07-26 20:45:21.720547...
Fetching data from 2025-05-31 20:45:21.720547 to 2025-06-28 20:45:21.720547...
Fetching data from 2025-05-03 20:45:21.720547 to 2025-05-31 20:45:21.720547...
Fetching data from 2025-04-05 20:45:21.720547 to 2025-05-03 20:45:21.720547...
Fetching data from 2025-03-08 20:45:21.720547 to 2025-04-05 20:45:21.720547...
Fetching data from 2025-02-08 20:45:21.720547 to 2025-03-08 20:45:21.720547...
Fetching data from 2025-01-11 20:45:21.720547 to 2025-02-08 20:45:21.720547...
Fetching data from 2024-12-14 20:45:21.720547 to 2025-01-11 20:45:21.720547...
Fetching data fr

Sensor_Name,Variable,Value,Timestamp,Flagged,Location_WKT,Sensor_Centroid_Longitude,Ground_Height_Above_Sea_Level,Sensor_Centroid_Latitude,Sensor_Height_Above_Ground,Broker_Name,Raw_ID
